# Retrieval crowd vs. AI crowd: a running-shoe buying assistant

We build the same shopping assistant three ways:

- **Path A:** LLM only. Stuff the catalog into the prompt and let the model answer.
- **Path B:** Retrieval only. Hybrid (BM25 + semantic) RRF over the catalog.
- **Path C:** Collaboration. Elastic Agent Builder calls retrieval as a tool and the LLM finishes.

We close with the `named_queries` pro tip: a deterministic gate that lets retrieval short circuit the LLM call when it is confident.

## Setup

In [22]:
%pip install -qU elasticsearch python-dotenv requests


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv
import os
import json
import requests

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Connect to Elasticsearch

In [24]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY, request_timeout=30
)
es_client.info()

ObjectApiResponse({'name': 'instance-0000000005', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.3.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '43a703737aab6baefa748bc7b69e4054926f2b2c', 'build_date': '2026-03-16T13:12:56.143057855Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Inference endpoints

Two roles. The embedding endpoint `.jina-embeddings-v5-text-small` is preconfigured by Elastic Inference Service, so we reference it directly in the index mapping with no PUT call. We create a chat completion endpoint backed by OpenAI.

| Role | Inference id | Service |
| :--- | :--- | :--- |
| Embeddings on `description_semantic` | `.jina-embeddings-v5-text-small` (preconfigured) | EIS |
| Chat completion | `shoes-chat` | OpenAI (`gpt-4o-mini`) |

In [ ]:
INFERENCE_ID = "shoes-chat"

try:
    es_client.inference.delete(inference_id=INFERENCE_ID)
except Exception:
    pass

es_client.inference.put(
    inference_id=INFERENCE_ID,
    task_type="chat_completion",
    body={
        "service": "openai",
        "service_settings": {
            "api_key": OPENAI_API_KEY,
            "model_id": "gpt-4o-mini",
        },
    },
)

print("Inference endpoint ready")

## Create the index

`description` is copied into a `semantic_text` field. `sku`, `surface`, `stability`, and `category` stay as `keyword` so they are queryable exactly. `name` stays as plain `text` for BM25 and constrained `match` clauses.

In [26]:
INDEX_NAME = "shoes"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "sku": {"type": "keyword"},
            "name": {"type": "text"},
            "aliases": {"type": "keyword"},
            "description": {
                "type": "text",
                "copy_to": "description_semantic",
            },
            "description_semantic": {
                "type": "semantic_text",
                "inference_id": ".jina-embeddings-v5-text-small",
            },
            "surface": {"type": "keyword"},
            "stability": {"type": "keyword"},
            "category": {"type": "keyword"},
        }
    },
)
print(f"Created index: {INDEX_NAME}")

Created index: shoes


## Bulk load the catalog

A small synthetic catalog (~20 running shoes) loaded from `dataset.json`. The catalog covers road and trail, neutral and stability, daily trainers, tempo, race, and trail categories.

In [27]:
from elasticsearch import helpers

with open("dataset.json") as f:
    shoes = json.load(f)

actions = ({"_index": INDEX_NAME, "_id": s["sku"], "_source": s} for s in shoes)
success, _ = helpers.bulk(es_client, actions, refresh=True)
print(f"Indexed {success} shoes")

Indexed 19 shoes


## Test queries

Two shapes the assistant has to handle:

- **Identification.** Resolve to one SKU.
- **Recommendation.** Return one shoe with a short reason.

In [28]:
Q_IDENTIFY = "Do you carry the Vaporfly 3 in men's 10, black?"
Q_RECOMMEND = (
    "a shoe for marathon training on asphalt, I mildly overpronate, "
    "prefer firmer cushioning"
)

## Path A: let the AI crowd win

Stuff the whole catalog into the prompt and ask the LLM to pick. Works at this size, breaks at 20,000 SKUs.

In [ ]:
def eis_chat(prompt: str) -> str:
    r = requests.post(
        f"{ELASTICSEARCH_URL}/_inference/chat_completion/shoes-chat/_stream",
        headers={
            "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
            "Content-Type": "application/json",
        },
        json={"messages": [{"role": "user", "content": prompt}]},
    )
    r.raise_for_status()

    text = ""
    for line in r.text.splitlines():
        if not line.startswith("data:"):
            continue
        chunk = line[len("data:") :].strip()
        if not chunk or chunk == "[DONE]":
            continue
        data = json.loads(chunk)
        for choice in data.get("choices", []):
            delta = choice.get("delta", {}).get("content")
            if delta:
                text += delta
    return text


def answer_llm_only(user_query: str) -> dict:
    catalog = [
        hit["_source"]
        for hit in es_client.search(
            index=INDEX_NAME,
            size=100,
            query={"match_all": {}},
            source_excludes=["description_semantic"],
        )["hits"]["hits"]
    ]

    prompt = f"""You are a running-shoe buying assistant.

Catalog (JSON):
{json.dumps(catalog, indent=2)}

User query: {user_query}

If the user is asking to identify a specific shoe, return JSON:
  {{"intent": "identify", "sku": "..."}}
If the user is asking for a recommendation, return JSON:
  {{"intent": "recommend", "sku": "...", "reason": "..."}}

Return ONLY the JSON object."""

    raw = eis_chat(prompt)
    cleaned = (
        raw.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )
    return json.loads(cleaned)


print("identify:", answer_llm_only(Q_IDENTIFY))
print("recommend:", answer_llm_only(Q_RECOMMEND))

identify: {'intent': 'identify', 'sku': 'NIKE-VPF3-M10-BLK'}
recommend: {'intent': 'recommend', 'sku': 'SAUC-GUI17-M10-RED', 'reason': 'The Saucony Guide 17 is purpose-built for marathon training on road surfaces like asphalt. Its CenterPath stability system addresses mild overpronation, and its PWRRUN foam delivers the firmer cushioning you prefer, making it well-suited for sustained marathon training blocks.'}


## Path B: let the retrieval crowd win

Hybrid RRF over a BM25 `match` on `name` and a `semantic` retriever on `description_semantic`. No LLM.

In [30]:
def answer_retrieval_only(user_query: str) -> dict:
    body = {
        "retriever": {
            "rrf": {
                "retrievers": [
                    {"standard": {"query": {"match": {"name": user_query}}}},
                    {
                        "standard": {
                            "query": {
                                "semantic": {
                                    "field": "description_semantic",
                                    "query": user_query,
                                }
                            }
                        }
                    },
                ],
                "rank_window_size": 20,
            }
        },
        "size": 5,
        "_source": ["sku", "name"],
    }
    hits = es_client.search(index=INDEX_NAME, body=body)["hits"]["hits"]
    return {"intent": "list", "hits": [h["_source"] for h in hits]}


print("identify:", json.dumps(answer_retrieval_only(Q_IDENTIFY), indent=2))
print("recommend:", json.dumps(answer_retrieval_only(Q_RECOMMEND), indent=2))

identify: {
  "intent": "list",
  "hits": [
    {
      "sku": "NIKE-VPF3-M10-ORG",
      "name": "Nike Vaporfly 3 Men's Orange Size 10"
    },
    {
      "sku": "NIKE-VPF3-M10-BLK",
      "name": "Nike Vaporfly 3 Men's Black Size 10"
    },
    {
      "sku": "ADID-AP3-M10-BLK",
      "name": "Adidas Adios Pro 3 Men's Black Size 10"
    },
    {
      "sku": "NB-SC4-M10-BLK",
      "name": "New Balance SuperComp Elite v4 Men's Black Size 10"
    },
    {
      "sku": "BRK-GH21-M10-BLK",
      "name": "Brooks Ghost 16 Men's Black Size 10"
    }
  ]
}
recommend: {
  "intent": "list",
  "hits": [
    {
      "sku": "SAUC-GUI17-M10-RED",
      "name": "Saucony Guide 17 Men's Red Size 10"
    },
    {
      "sku": "ASICS-KAY30-M10-GRY",
      "name": "Asics Kayano 30 Men's Grey Size 10"
    },
    {
      "sku": "BRK-AD23-M10-BLU",
      "name": "Brooks Adrenaline GTS 23 Men's Blue Size 10"
    },
    {
      "sku": "HOKA-ARA7-M10-BLK",
      "name": "Hoka Arahi 7 Men's Black Size 10"
   

## Path C: the collaboration with Agent Builder

Declare an `index_search` tool over `shoes`, register an agent that pairs it with `shoes-chat`, and call `converse`. The agent retrieves and the LLM finishes.

In [31]:
headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

In [32]:
TOOL_ID = "search_shoes"
AGENT_ID = "shoe-assistant"

requests.delete(f"{KIBANA_URL}/api/agent_builder/tools/{TOOL_ID}", headers=headers)

r = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json={
        "id": TOOL_ID,
        "type": "index_search",
        "description": (
            "Search the shoes catalog. Returns up to 5 ranked candidates with "
            "sku, name, description, surface, stability, and category."
        ),
        "configuration": {"pattern": INDEX_NAME, "row_limit": 5},
    },
)
r.raise_for_status()
print("Tool created")

Tool created


In [33]:
requests.delete(f"{KIBANA_URL}/api/agent_builder/agents/{AGENT_ID}", headers=headers)

r = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json={
        "id": AGENT_ID,
        "name": "Shoe Assistant",
        "description": (
            "Help shoppers identify a shoe or get a recommendation. Use "
            "search_shoes to find candidates and answer in one or two "
            "sentences."
        ),
        "configuration": {
            "tools": [{"tool_ids": [TOOL_ID]}],
            "instructions": (
                "Always call search_shoes first. For identification queries "
                "return a single SKU. For recommendation queries return one "
                "SKU and a short reason."
            ),
        },
    },
)
r.raise_for_status()
print("Agent created")

Agent created


In [34]:
def converse(user_input: str) -> str:
    r = requests.post(
        f"{KIBANA_URL}/api/agent_builder/converse",
        headers=headers,
        json={"agent_id": AGENT_ID, "input": user_input},
    )
    r.raise_for_status()
    return r.json()


print("identify:", converse(Q_IDENTIFY))
print("recommend:", converse(Q_RECOMMEND))

identify: {'conversation_id': '815deece-ed0d-405f-ac02-1fde0872982a', 'steps': [{'type': 'reasoning', 'reasoning': 'The user is asking a product identification question, so I should use the search_shoes tool to find the relevant product information.'}, {'type': 'tool_call', 'tool_call_id': 'tool_search_shoes_3lKtXKT7V1ZxWkTXQslk', 'tool_id': 'search_shoes', 'progression': [{'message': 'Identifying the most relevant data source'}, {'message': 'Analyzing strategy to search against "shoes"'}, {'message': 'Searching documents for "Vaporfly 3 men\'s 10, black"'}], 'params': {'nlQuery': "Vaporfly 3 men's 10, black"}, 'results': [{'tool_result_id': 'i88Nul', 'type': 'resource', 'data': {'reference': {'id': 'NIKE-VPF3-M10-ORG', 'index': 'shoes'}, 'partial': True, 'content': {'highlights': ["Nike <em>Vaporfly</em> <em>3</em> <em>Men's</em> Orange Size <em>10</em>", 'Carbon-plate racing shoe with ZoomX foam. Race-day road shoe in volt orange, responsive and light.']}}}, {'tool_result_id': 'KwgIB

## Pro tip: `named_queries` as a deterministic gate

Tag a *narrow* clause with `_name`. When `matched_queries` lists that name, retrieval is confident enough to short circuit the LLM call. Loose clauses do not work as gates: a single-token `match` on `name` AND-matches every brand shoe and `matched_queries` becomes noise.

Below, `name_match` only fires when at least three of the user's content words land in the product name. The semantic clause stays unnamed: its job is ranking, not routing.

In [35]:
def gated_search(user_query: str) -> dict:
    body = {
        "query": {
            "bool": {
                "should": [
                    {
                        "match": {
                            "name": {
                                "query": user_query,
                                "minimum_should_match": 3,
                                "_name": "name_match",
                            }
                        }
                    },
                    {
                        "semantic": {
                            "field": "description_semantic",
                            "query": user_query,
                        }
                    },
                ]
            }
        },
        "size": 3,
        "_source": ["sku", "name"],
    }
    return es_client.search(index=INDEX_NAME, body=body)


def route(user_query: str) -> dict:
    res = gated_search(user_query)
    hits = res["hits"]["hits"]
    if hits and "name_match" in hits[0].get("matched_queries", []):
        return {
            "intent": "identify",
            "sku": hits[0]["_source"]["sku"],
            "source": "retrieval (gate fired)",
        }
    return {
        "intent": "recommend",
        "candidates": [h["_source"] for h in hits],
        "source": "forward to LLM",
    }


print("identify:", json.dumps(route("vaporfly 3 men's 10 black"), indent=2))
print("recommend:", json.dumps(route(Q_RECOMMEND), indent=2))

identify: {
  "intent": "identify",
  "sku": "NIKE-VPF3-M10-BLK",
  "source": "retrieval (gate fired)"
}
recommend: {
  "intent": "recommend",
  "candidates": [
    {
      "sku": "SAUC-GUI17-M10-RED",
      "name": "Saucony Guide 17 Men's Red Size 10"
    },
    {
      "sku": "ASICS-KAY30-M10-GRY",
      "name": "Asics Kayano 30 Men's Grey Size 10"
    },
    {
      "sku": "BRK-AD23-M10-BLU",
      "name": "Brooks Adrenaline GTS 23 Men's Blue Size 10"
    }
  ],
  "source": "forward to LLM"
}


## Cleanup

In [36]:
es_client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)

requests.delete(f"{KIBANA_URL}/api/agent_builder/agents/{AGENT_ID}", headers=headers)
requests.delete(f"{KIBANA_URL}/api/agent_builder/tools/{TOOL_ID}", headers=headers)

try:
    es_client.inference.delete(inference_id="shoes-chat")
except Exception as e:
    print(f"shoes-chat: {e}")

print("Cleaned up")

Cleaned up
